In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-python-module/docker-compose.yml down minio

In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-python-module/docker-compose.yml up -d minio


#### Delete Existing Delta Table

In [1]:
%%bash

## Delete Existing Delta Table
aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/ --recursive

## Delete Existing Delta Table
aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/spark/ --recursive


In [1]:
from pyspark_module.utils.SparkSessionManager import SparkSessionManager

spark = SparkSessionManager("pyspark-app").create_session(type="spark")

26/07/15 09:44:54 WARN Utils: Your hostname, sandbox-kubuntu resolves to a loopback address: 127.0.1.1; using 192.168.70.128 instead (on interface ens33)
26/07/15 09:44:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/brijeshdhaker/IdeaProjects/bd-python-module/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brijeshdhaker/.ivy2/cache
The jars for the packages stored in: /home/brijeshdhaker/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c63525c-fef9-4d80-9b30-b628c92bc283;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 189ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	or

In [2]:
%load_ext sql

Loading configurations from /home/brijeshdhaker/IdeaProjects/bd-python-module/pyproject.toml.

Settings changed:

Config,value
feedback,True
autopandas,True


In [3]:
%sql spark

#### Create Database

In [ ]:
%%sql

ALTER DATABASE default SET LOCATION 's3://warehouse/default/spark'

In [4]:
%%sql

USE spark_catalog.default;

Running query in 'SparkSession'

""


In [5]:
%%sql 

DESCRIBE SCHEMA spark_catalog.default;

Running query in 'SparkSession'

5 rows affected.

,info_name,info_value
0,Catalog Name,spark_catalog
1,Namespace Name,default
2,Comment,Default Hive database
3,Location,s3a://warehouse/default/spark
4,Owner,public


In [6]:
%%sql 

show tables in spark_catalog.default;

Running query in 'SparkSession'

,namespace,tableName,isTemporary


#### Spark Tables

In [7]:
%%sql 

--
DROP TABLE IF EXISTS spark_catalog.default.spark_table;

--
CREATE TABLE spark_catalog.default.spark_table (
    id INT,
    firstName STRING,
    middleName STRING,
    lastName STRING,
    gender STRING,
    birthDate TIMESTAMP,
    ssn STRING,
    salary INT
)
--LOCATION 's3a://warehouse/default/spark/spark_table'

Running query in 'SparkSession'

""


In [8]:
%%sql 

show tables in spark_catalog.default;

Running query in 'SparkSession'

1 rows affected.

,namespace,tableName,isTemporary
0,default,spark_table,False


In [9]:
%%sql 

show create table spark_catalog.default.spark_table;

Running query in 'SparkSession'

1 rows affected.

,createtab_stmt
0,"CREATE TABLE default.spark_table (\n id INT,\..."


In [ ]:
%%sql 

DESCRIBE TABLE FORMATTED spark_catalog.default.spark_table

In [ ]:
%%sql 

DESCRIBE HISTORY spark_catalog.default.spark_table;

#### Delta Tables

In [11]:
%%sql 

show SCHEMAS;

Running query in 'SparkSession'

1 rows affected.

,namespace
0,default


In [12]:
%%sql

DROP SCHEMA IF EXISTS deltalake;

Running query in 'SparkSession'

""


In [13]:
%%sql
-- In Spark SQL, the terms DATABASE and SCHEMA are interchangeable synonyms;
CREATE DATABASE IF NOT EXISTS deltalake
COMMENT 'This is a default database for the deltalake'
LOCATION 's3a://warehouse/default/deltalake';

Running query in 'SparkSession'

""


In [ ]:
%%sql

CREATE SCHEMA IF NOT EXISTS deltalake
COMMENT 'This is a default database for the deltalake'
LOCATION 's3a://warehouse/default/deltalake';


In [14]:
%%sql 

show SCHEMAS;

Running query in 'SparkSession'

2 rows affected.

,namespace
0,default
1,deltalake


In [15]:
%%sql 

DESC SCHEMA deltalake;

Running query in 'SparkSession'

5 rows affected.

,info_name,info_value
0,Catalog Name,spark_catalog
1,Namespace Name,deltalake
2,Comment,This is a default database for the deltalake
3,Location,s3a://warehouse/default/deltalake
4,Owner,brijeshdhaker


In [16]:
%%sql 

DESCRIBE DATABASE deltalake;

Running query in 'SparkSession'

5 rows affected.

,info_name,info_value
0,Catalog Name,spark_catalog
1,Namespace Name,deltalake
2,Comment,This is a default database for the deltalake
3,Location,s3a://warehouse/default/deltalake
4,Owner,brijeshdhaker


In [ ]:
%%sql

ALTER DATABASE deltalake SET LOCATION 's3a://warehouse/default/deltalake'

In [17]:
%%sql

USE deltalake;

Running query in 'SparkSession'

""


In [18]:
%%sql 

--
DROP TABLE IF EXISTS spark_catalog.deltalake.delta_table;

--
CREATE TABLE IF NOT EXISTS spark_catalog.deltalake.delta_table (
    id INT,
    firstName STRING,
    middleName STRING,
    lastName STRING,
    gender STRING,
    birthDate TIMESTAMP,
    ssn STRING,
    salary INT
)
USING DELTA
--LOCATION 's3a://warehouse/default/deltalake/delta_table'

Running query in 'SparkSession'

""


In [19]:
%%sql 

show tables in spark_catalog.deltalake;

Running query in 'SparkSession'

1 rows affected.

,namespace,tableName,isTemporary
0,deltalake,delta_table,False


In [20]:
%%sql 

DESCRIBE TABLE FORMATTED spark_catalog.deltalake.delta_table

Running query in 'SparkSession'

16 rows affected.

,col_name,data_type,comment
0,id,int,NaN
1,firstName,string,NaN
2,middleName,string,NaN
3,lastName,string,NaN
4,gender,string,NaN
5,birthDate,timestamp,NaN
6,ssn,string,NaN
7,salary,int,NaN
8,,,
9,# Detailed Table Information,,


In [21]:
%%sql 

DESCRIBE TABLE EXTENDED spark_catalog.deltalake.delta_table;

Running query in 'SparkSession'

16 rows affected.

,col_name,data_type,comment
0,id,int,NaN
1,firstName,string,NaN
2,middleName,string,NaN
3,lastName,string,NaN
4,gender,string,NaN
5,birthDate,timestamp,NaN
6,ssn,string,NaN
7,salary,int,NaN
8,,,
9,# Detailed Table Information,,


In [22]:
%%sql 

DESCRIBE HISTORY spark_catalog.deltalake.delta_table;

Running query in 'SparkSession'

1 rows affected.

,version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,0,2026-07-15 09:48:24,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'pr...",None,None,None,NaN,Serializable,True,{},None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [23]:
%%sql 

DESCRIBE DETAIL spark_catalog.deltalake.delta_table;

Running query in 'SparkSession'

1 rows affected.

,format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures
0,delta,6dada067-c251-46fd-930f-3c1e744e0610,spark_catalog.deltalake.delta_table,None,s3a://warehouse/default/deltalake/delta_table,2026-07-15 09:48:23.947,2026-07-15 09:48:24,[],[],0,0,{},1,2,"[appendOnly, invariants]"


##### List catalogs in the current/default database

In [ ]:
# Get a list of all catalogs
catalogs = spark.catalog.listCatalogs()

# Iterate and print each catalog name
print("Available catalogs:")
for catalog in catalogs:
    # The 'catalog' object has a 'name' attribute
    print(f"* {catalog.name}") 

In [ ]:
# Get the name of the current catalog
current_catalog = spark.catalog.currentCatalog()
print(f"Current catalog name: {current_catalog}")

##### List databases in the current/default catalog

In [ ]:

# List databases in the current/default catalog
databases = spark.catalog.listDatabases()
for db in databases :
    print(f"- {db}")

##### List tables in the current/default database

In [ ]:

# List tables in the current/default database
tables_list = spark.catalog.listTables()
print(f"Tables in all database")
for table in tables_list:
    print(f" namespace - {table.namespace}  table - {table.name}")

print()
print("-"*50)
print()
# List tables in a specific database
tables_in_db = spark.catalog.listTables("default")
print(f"Tables in default database")

# Iterate and print table names (optional)
for table in tables_in_db:
    print(f" namespace - {table.namespace}  table - {table.name}")

In [ ]:
# The spark.catalog object provides many more functions for metadata management: 

# Returns a list of columns and their data types.
spark.catalog.listColumns("delta_table")

# Caches the table data in memory for faster access.
spark.catalog.cacheTable("delta_table")

# Removes the table from the cache.
spark.catalog.uncacheTable("delta_table")

# Checks if a table exists.
spark.catalog.tableExists("delta_table")

##### Save the DataFrame as a table in the spark default database

In [ ]:
# Create a sample DataFrame
data = [("James", 34), ("Margaret", 28), ("Robert", 45)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)

# Save the DataFrame as a table in the current database
df.write.saveAsTable("spark_catalog.default.people_table")

##### Create an Managed table:

In [ ]:
# Creates an empty managed table with a defined schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])
spark.catalog.createTable("managed_people_table", schema=schema, source="parquet")

##### Create an External table:

In [ ]:
# Writing a DataFrame to a specific location as an external table
df.write \
  .option("path", "s3a://warehouse/default/spark/my_external_parquet_table") \
  .format("parquet") \
  .saveAsTable("my_external_parquet_table")

##### Create an External table:


In [ ]:
# Define a path where data is stored (e.g., in CSV format)
file_path = "s3a://warehouse/default/spark/my_external_parquet_table"

# Create an external table from the data at the given path
# Registers existing CSV data as an external table
spark.catalog.createTable(
    "external_people_table", 
    path="s3a://warehouse/default/spark/my_external_parquet_table", 
    source="parquet", 
    header="true", 
    inferSchema="true"
)



In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", TimestampType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

peoples_df = spark.read.format("csv").option("header", True).schema(schema).load("s3a://datasets/people-data/peoples.csv")
peoples_df.printSchema()

#
display(peoples_df)

In [ ]:
peoples_df.write.mode("append").saveAsTable("spark_catalog.default.spark_table")

In [ ]:
# If you know the table does not already exist, you can call this instead:
peoples_df.write.mode("append").insertInto("spark_catalog.default.spark_table")

##### Save the DataFrame as a table in the deltalake database

In [ ]:
# Create the table if it does not exist. Otherwise, replace the existing table.
peoples_df.writeTo("spark_catalog.deltalake.delta_table").createOrReplace()

In [ ]:
# Or to a named table:
peoples_df.write.format("delta").mode("append").saveAsTable("spark_catalog.deltalake.delta_table")

In [ ]:
# Save as delta table into Minio S3
peoples_df.write.format('delta').mode("append").save('s3a://warehouse/default/deltalake/delta_table')
